# MP23 — Channel Attention from Scratch

Channel attention summarizes each feature channel and learns a multiplier. Feature channels are learned maps, not necessarily physical sensor bands.

## Mathematics and worked example

For F with shape N×C×H×W, z[n,c]=mean(F[n,c,:,:]). Two 1×1 convolutions implement sigmoid(W2 ReLU(W1 z)). The gate has shape N×C×1×1 and broadcasts spatially. If a channel is [2,4], its mean is 3; a gate of 0.5 returns [1,2]. Width 8 with bottleneck 2 has (8×2+2)+(2×8+8)=42 channel-gate parameters. Disabled means multiplier 1; uniform means fixed 0.5, not a learned distribution.

In [ ]:
from pathlib import Path
import sys,json,torch
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/rssr').is_dir())
sys.path.insert(0,str(ROOT/'src'))
from mp23_tools import InspectableAttention
torch.manual_seed(23)
x=torch.tensor([[[[2.,4.]]]*8])
g=InspectableAttention(8)
y,c,s=g(x,'uniform')
assert c.shape==(1,8,1,1)
torch.testing.assert_close(y,x/2)
assert sum(p.numel() for p in g.channel.parameters())==42
print(c.flatten())

## Executed experiment and reproducibility

Run `python scripts/attention_lessons_experiment.py` from the repository root. This is same-grid RGB reconstruction, not super-resolution. A tiny shape fixture precedes training on four NAIP scenes, with four separate validation scenes, 16 fixed 32×32 patches each. USDA/OpenSR NAIP imagery is public domain; values are stored DN/10000, not claimed calibrated reflectance. No new imagery download or final-test access is required.

CPU float32, two threads, seed 23, Adam 0.001, batch four, 300 updates, MSE. Models share identical head/tail initialization and minibatch sequence. Evaluate the fixed last step, with no checkpoint selection. Validation PSNR uses range one, border two, equal-size patches and equal scene weighting. Inactive gate modules remain allocated to preserve initialization; the report separately counts active and allocated parameters. Inference timing uses five warmups and twenty observations on a 32×32 input; it is not whole-app latency.

The first figure compares separately trained reconstructions under one reference contrast scale. The second applies alternative gate modes to identical features from the trained combined model, plus synthetic step-edge/checkerboard fixtures. All mask heatmaps share [0,1]; these are not semantic labels. Training curves show sampled minibatch loss, not a full-dataset estimate. Read the actual measured table below; preserve unfavorable outcomes.


In [ ]:
r=json.loads((ROOT/'reports/mp23-mp25-attention.json').read_text())
for row in r['results']:
 print({k:v for k,v in row.items() if k!='per_tile'})
from IPython.display import display,Image
for name in ['mp23-mp25-reconstruction.png','mp23-mp25-attention.png']:
 display(Image(filename=str(ROOT/'reports'/name)))

## Practice

1. Does pooling retain the location of the largest feature?
2. Why compare a fixed 0.5 gate?
3. Are gates probabilities over channels?

## Worked solutions

1. No. Spatial averaging discards location.
2. It separates simple attenuation from learned input-dependent weights.
3. No. Independent sigmoid gates need not sum to one.

## Limitations

One seed and a short reconstruction experiment cannot establish SR superiority, causality, explainability or geographic generalization. Equal update count is not equal computation. Learner mastery is unassessed. Frozen research weights and test results are untouched.

[Authoritative brief](README.md) · [Local/Colab setup](../../README.md)